# 06 — CNN-LSTM Comparison: 1D DWT vs 2D Kurtogram

**Approach A (Ours):** DWT time series → 1D CNN + ResBlock + ChannelAttn + LSTM

**Approach B (Paper):** Kurtogram image → 2D CNN + ResBlock + ChannelAttn + LSTM

**Reference:** 'Development of hybrid CNN-LSTM for NILM' ResearchGate 2025

**Goal:** Prove that DWT 1D approach is better than Kurtogram 2D approach

**Author:** Chadha Jeddi — NILM Benchmarking Project

## 1. Setup

In [ ]:
import os, sys, glob, time, json
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_DIR = '/kaggle/working/nilm-benchmarking'
SAVE_DIR = '/kaggle/working/nilm_results'
CKPT_DIR = f'{SAVE_DIR}/checkpoints'
RES_DIR  = f'{SAVE_DIR}/results'
for d in [SAVE_DIR, CKPT_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
for p in ['src','models','models/baselines','models/proposed']:
    sys.path.insert(0, f'{REPO_DIR}/{p}')

for d in ['data/raw/UK-DALE','data/raw/UKDALE','data/processed']:
    os.makedirs(d, exist_ok=True)

ukdale = glob.glob('/kaggle/input/**/ukdale.h5', recursive=True)
if ukdale:
    for dst in ['data/raw/UK-DALE/ukdale.h5','data/raw/UKDALE/ukdale.h5']:
        if not os.path.exists(dst): os.symlink(ukdale[0], dst)
    print(f'OK: {ukdale[0]}')

for f in glob.glob('data/processed/*.parquet'): os.remove(f)

os.system('pip install -q PyWavelets pyarrow h5py tqdm einops torchinfo seaborn scipy')

import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')
import torch.nn as nn, torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


## 2. Imports & Config

In [ ]:
from config import WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES, APPLIANCE_NAMES, APPLIANCES
from preprocessing import load_ukdale_house, preprocess_house
from dataset import load_clean_df, save_clean_df, split_train_val, build_dataloaders
from metrics import MetricsTracker, focal_loss
from train import EarlyStopping
from cnn_lstm_model import CNNLSTMBaseline        # Approach A: 1D DWT
from cnn_lstm_2d_model import (CNNLSTMKurtogram,  # Approach B: 2D Kurtogram
                               compute_kurtogram_tensor)

# Shared config
LR           = 1e-3
WEIGHT_DECAY = 1e-2
DROPOUT      = 0.2
EPOCHS       = 100
PATIENCE     = 25
SCHED_PAT    = 15

COLORS = {'kettle':'#D94040','fridge':'#2E9E5A','washing_machine':'#E8922A',
          'dishwasher':'#7B4FBF','microwave':'#CC3399'}
app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
print('Config loaded')


## 3. Data Loading

In [ ]:
print('Loading UK-DALE House 1...')
cached = load_clean_df('UK-DALE', 1)
if cached is not None:
    clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

# Approach A: 1D DWT DataLoader
train_loader_1d, val_loader_1d, norm_stats = build_dataloaders(
    train_df, val_df, batch_size=256, train_stride=30,
    val_stride=480, num_workers=2, instance_norm=True, seg_size=None)

x_s, yp_s, ys_s = next(iter(val_loader_1d))
print(f'Approach A batch: x={tuple(x_s.shape)} y={tuple(yp_s.shape)}')


## 4. Kurtogram Dataset (Approach B)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dwt import dwt_transform
import numpy as np

class KurtogramDataset(Dataset):
    """
    Converts each power window to a 6×8×24 Kurtogram tensor.
    6 channels (DWT sub-bands) × 8 frequency levels × 24 time steps.
    This follows the paper's approach of using 2D images as input.
    """
    def __init__(self, df, stride=30):
        self.df = df
        self.agg = df['aggregate'].values.astype(np.float32)
        self.starts = list(range(0, len(df)-WINDOW_SIZE+1, stride))
        # Precompute targets
        self.power = np.stack([
            np.nan_to_num(df[a].values/APPLIANCES[a]['max_power'], nan=0.0).clip(0,1)
            for a in APPLIANCE_NAMES], axis=1).astype(np.float32)
        self.state = np.stack([
            np.nan_to_num(df[f'{a}_state'].values, nan=0.0)
            for a in APPLIANCE_NAMES], axis=1).astype(np.float32)

    def __len__(self): return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        c = s + WINDOW_SIZE // 2
        window = self.agg[s:s+WINDOW_SIZE]

        # Instance normalize
        w_mean = window.mean(); w_std = window.std() + 1e-8
        window_norm = (window - w_mean) / w_std

        # Get DWT sub-bands
        dwt = dwt_transform(window_norm)  # (4, 480)

        # Compute Kurtogram for each DWT sub-band + 2 temporal
        hour = self.df.index[c].hour
        sin_h = np.full(WINDOW_SIZE, np.sin(2*np.pi*hour/24), dtype=np.float32)
        cos_h = np.full(WINDOW_SIZE, np.cos(2*np.pi*hour/24), dtype=np.float32)
        all_channels = np.concatenate([dwt, sin_h[None], cos_h[None]], axis=0)  # (6,480)

        # Convert each channel to Kurtogram (8×24)
        kurtograms = np.stack([
            compute_kurtogram_tensor(all_channels[ch], n_levels=8, n_time=24)
            for ch in range(INPUT_CHANNELS)
        ], axis=0)  # (6, 8, 24)

        y_power = self.power[c]  # (5,)
        y_state = self.state[c]  # (5,)

        return (torch.tensor(kurtograms),
                torch.tensor(y_power),
                torch.tensor(y_state))

print('Building Kurtogram datasets (this takes ~2-3 minutes)...')
print('Each window is converted to 6×8×24 Kurtogram images...')

# Use larger stride for Kurtogram (expensive to compute)
train_ds_2d = KurtogramDataset(train_df, stride=120)  # stride=120 for speed
val_ds_2d   = KurtogramDataset(val_df,   stride=480)

train_loader_2d = DataLoader(train_ds_2d, batch_size=64,
                              shuffle=True, num_workers=2)
val_loader_2d   = DataLoader(val_ds_2d,   batch_size=64,
                              shuffle=False, num_workers=2)

x_k, yp_k, ys_k = next(iter(val_loader_2d))
print(f'Approach B batch: x={tuple(x_k.shape)} y={tuple(yp_k.shape)}')
print(f'Each image: {tuple(x_k.shape[1:])} = (channels, freq_levels, time_steps)')


## 5. Training Function (reused for both)

In [ ]:
def train_model(model, train_loader, val_loader, model_name,
                epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY,
                patience=PATIENCE, sched_pat=SCHED_PAT):
    ckpt_path = f'{CKPT_DIR}/{model_name}_best.pth'
    hist_path = f'{RES_DIR}/{model_name}_history.json'

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=sched_pat)
    stopper = EarlyStopping(patience=patience)
    tracker = MetricsTracker(APPLIANCE_NAMES, app_max)

    history = {'epoch':[],'train_loss':[],'val_loss':[],
               'val_mr':[],'val_f1':[],'val_mae':[],'lr':[]}
    best_mr, best_state, best_epoch = -float('inf'), None, 0
    n_params = sum(p.numel() for p in model.parameters())
    start = time.time()

    print(f'Training {model_name} | {n_params:,} params', flush=True)
    print('='*70, flush=True)

    for epoch in range(1, epochs+1):
        ep_start = time.time()

        # TRAIN
        model.train()
        total_tl = 0; nb_t = 0
        for x, y_power, y_state in train_loader:
            x = x.to(DEVICE)
            y_power = y_power.to(DEVICE)
            y_state = y_state.to(DEVICE)

            # Augmentation — scale only (noise on Kurtogram would distort image)
            scale = (0.7 + torch.rand(x.shape[0], 1,
                    *([1]*(x.dim()-2))) * 0.6).to(DEVICE)
            x = x * scale
            y_power = (y_power * scale.view(x.shape[0],-1)[:,:1]).clamp(0,1)

            pred_p, pred_s, pred_g = model(x)

            loss_all = F.smooth_l1_loss(pred_p, y_power)
            mask = (y_state > 0.5)
            if mask.sum() > 0:
                loss_act = F.smooth_l1_loss(pred_p[mask], y_power[mask])
                loss_p = 0.5*loss_all + 0.5*loss_act
            else:
                loss_p = loss_all
            loss_s = focal_loss(pred_s, y_state, alpha=0.75, gamma=2.0)
            loss_g = F.smooth_l1_loss(pred_g, y_power)
            loss = 1.0*loss_p + 2.0*loss_s + 0.5*loss_g

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_tl += loss.item(); nb_t += 1
        train_loss = total_tl / max(nb_t, 1)

        # VALIDATE
        model.eval()
        total_vl = 0; nb_v = 0; tracker.reset()
        with torch.no_grad():
            for x, yp, ys in val_loader:
                x = x.to(DEVICE); yp = yp.to(DEVICE); ys = ys.to(DEVICE)
                pred_p, pred_s, pred_g = model(x)
                mask = (ys > 0.5)
                la = F.smooth_l1_loss(pred_p, yp)
                if mask.sum() > 0:
                    lact = F.smooth_l1_loss(pred_p[mask], yp[mask])
                    lp = 0.5*la + 0.5*lact
                else: lp = la
                ls = focal_loss(pred_s, ys, alpha=0.75, gamma=2.0)
                lg = F.smooth_l1_loss(pred_g, yp)
                total_vl += (1.0*lp+2.0*ls+0.5*lg).item(); nb_v += 1
                tracker.update(pred_p.cpu(), yp.cpu(), pred_s.cpu(), ys.cpu())

        val_loss = total_vl / max(nb_v, 1)
        m = tracker.compute()
        val_mr = m['mean']['mr']; val_f1 = m['mean']['f1']
        val_mae = m['mean']['mae_w']
        cur_lr = opt.param_groups[0]['lr']
        ep_time = time.time() - ep_start

        for k, v in zip(['epoch','train_loss','val_loss','val_mr','val_f1','val_mae','lr'],
                        [epoch,train_loss,val_loss,val_mr,val_f1,val_mae,cur_lr]):
            history[k].append(v)

        is_best = val_mr > best_mr
        if is_best:
            best_mr=val_mr; best_epoch=epoch
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
            torch.save({'model_state_dict':best_state,'best_epoch':best_epoch,
                        'best_val_mr':best_mr,'model_name':model_name}, ckpt_path)

        if epoch%5==0 or is_best:
            with open(hist_path,'w') as f:
                json.dump({**history,'best_epoch':best_epoch,'best_val_mr':best_mr,
                           'model':model_name,'time':time.time()-start},f)

        star=' *' if is_best else ''
        print(f'Ep {epoch:3d}/{epochs} | L:{train_loss:.4f}/{val_loss:.4f} | '
              f'MR:{val_mr:.3f} F1:{val_f1:.3f} MAE:{val_mae:.1f}W | '
              f'LR:{cur_lr:.1e} | {ep_time:.0f}s{star}', flush=True)

        sched.step(val_mr)
        if stopper.step(val_mr):
            print(f'Early stopping at epoch {epoch}', flush=True); break

    total_time = time.time()-start
    print(f'\nDone {total_time/60:.1f}min | Best ep{best_epoch} MR={best_mr:.4f}', flush=True)
    model.load_state_dict(best_state)
    return model, history, best_mr, best_epoch, total_time

print('Training function ready')


## 6. Train Approach A — 1D DWT CNN-LSTM
Conv1D + ResidualBlock + ChannelAttention + LSTM

In [ ]:
print('APPROACH A: 1D DWT → CNN-LSTM + ResBlock + ChannelAttn')
print('Input: (B, 6, 480) — DWT time series')
print()

model_a = CNNLSTMBaseline(
    in_channels=INPUT_CHANNELS, window_size=WINDOW_SIZE,
    n_appliances=N_APPLIANCES, dropout=DROPOUT).to(DEVICE)

n_a = sum(p.numel() for p in model_a.parameters())
print(f'Parameters: {n_a:,} | INT8: {n_a/1e6:.3f} MB')

model_a, hist_a, mr_a, ep_a, time_a = train_model(
    model_a, train_loader_1d, val_loader_1d, 'cnn_lstm_1d')


## 7. Train Approach B — 2D Kurtogram CNN-LSTM
Kurtogram images → 2D CNN + ResidualBlock + ChannelAttention + LSTM

In [ ]:
print('APPROACH B: Kurtogram → 2D CNN-LSTM + ResBlock + ChannelAttn')
print('Input: (B, 6, 8, 24) — Kurtogram images')
print()

model_b = CNNLSTMKurtogram(
    in_channels=INPUT_CHANNELS, n_appliances=N_APPLIANCES,
    dropout=DROPOUT).to(DEVICE)

n_b = sum(p.numel() for p in model_b.parameters())
print(f'Parameters: {n_b:,} | INT8: {n_b/1e6:.3f} MB')

model_b, hist_b, mr_b, ep_b, time_b = train_model(
    model_b, train_loader_2d, val_loader_2d, 'cnn_lstm_2d')


## 8. Results Comparison

In [ ]:
# Evaluate both on val set
def evaluate(model, val_loader):
    model.eval()
    tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
    tracker.reset()
    all_pp, all_tp, all_ps, all_ts = [], [], [], []
    with torch.no_grad():
        for x, yp, ys in val_loader:
            pp, ps, pg = model(x.to(DEVICE))
            pp=pp.cpu(); ps=ps.cpu()
            tracker.update(pp, yp, ps, ys)
            all_pp.append(pp.numpy()); all_tp.append(yp.numpy())
            all_ps.append((torch.sigmoid(ps)>=0.5).float().numpy())
            all_ts.append(ys.numpy())
    metrics = tracker.compute()
    return (metrics,
            np.concatenate(all_pp), np.concatenate(all_tp),
            np.concatenate(all_ps), np.concatenate(all_ts))

print('Evaluating Approach A (1D DWT)...')
m_a, pp_a, tp_a, ps_a, ts_a = evaluate(model_a, val_loader_1d)

print('Evaluating Approach B (2D Kurtogram)...')
m_b, pp_b, tp_b, ps_b, ts_b = evaluate(model_b, val_loader_2d)

# Print comparison
print(f'\n{"="*70}')
print('COMPARISON: Approach A (1D DWT) vs Approach B (2D Kurtogram)')
print(f'{"="*70}')
print(f'{"Metric":<15} {"1D DWT":>12} {"2D Kurtogram":>15} {"Winner":>10}')
print('-'*55)
for metric, higher_better in [('mr',True),('f1',True),('mae_w',False),
                               ('nde',False),('teca',True)]:
    v_a = m_a['mean'][metric]
    v_b = m_b['mean'][metric]
    if higher_better:
        winner = 'A (DWT)' if v_a > v_b else 'B (Kurt)'
    else:
        winner = 'A (DWT)' if v_a < v_b else 'B (Kurt)'
    print(f'{metric.upper():<15} {v_a:>12.4f} {v_b:>15.4f} {winner:>10}')
print()
print(f'Training time A: {time_a/60:.1f} min | B: {time_b/60:.1f} min')
print(f'Parameters    A: {sum(p.numel() for p in model_a.parameters()):,} | '
      f'B: {sum(p.numel() for p in model_b.parameters()):,}')

# Save comparison
comparison = {
    'approach_a_1d_dwt': {'metrics': m_a, 'best_epoch': ep_a,
                          'best_mr': mr_a, 'training_time_min': time_a/60},
    'approach_b_2d_kurtogram': {'metrics': m_b, 'best_epoch': ep_b,
                                'best_mr': mr_b, 'training_time_min': time_b/60},
}
with open(f'{RES_DIR}/cnn_lstm_comparison.json','w') as f:
    json.dump(comparison, f, indent=2)
print('Comparison saved: cnn_lstm_comparison.json')


## 9. Comparison Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_list = ['mr', 'f1', 'mae_w']
labels = ['MR ↑', 'F1 ↑', 'MAE (W) ↓']

for ax_i, (metric, label) in enumerate(zip(metrics_list, labels)):
    vals_a = [m_a[a][metric] for a in APPLIANCE_NAMES]
    vals_b = [m_b[a][metric] for a in APPLIANCE_NAMES]

    x = np.arange(N_APPLIANCES)
    width = 0.35
    bars_a = axes[ax_i].bar(x - width/2, vals_a, width,
                             label='A: 1D DWT', color='#3366CC', alpha=0.8)
    bars_b = axes[ax_i].bar(x + width/2, vals_b, width,
                             label='B: 2D Kurtogram', color='#D94040', alpha=0.8)
    axes[ax_i].set_title(label, fontsize=12)
    axes[ax_i].set_xticks(x)
    axes[ax_i].set_xticklabels(APPLIANCE_NAMES, rotation=20, ha='right')
    axes[ax_i].legend(fontsize=9)

    for bar, val in zip(list(bars_a)+list(bars_b), vals_a+vals_b):
        axes[ax_i].text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()*1.02,
                        f'{val:.3f}', ha='center', fontsize=7)

plt.suptitle(
    'CNN-LSTM Comparison: 1D DWT (Ours) vs 2D Kurtogram (Paper)\n'
    f'Blue=1D DWT (MR={mr_a:.3f})  Red=2D Kurtogram (MR={mr_b:.3f})',
    fontsize=13
)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/cnn_lstm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_lstm_comparison.png')


## 10. Cross-House Evaluation — House 2

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dwt import dwt_transform

class CrossHouseDataset1D(Dataset):
    def __init__(self, df, stride=480):
        self.df = df
        self.starts = list(range(0, len(df)-WINDOW_SIZE+1, stride))
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]; c = s + WINDOW_SIZE//2
        window = self.df['aggregate'].values[s:s+WINDOW_SIZE].astype(np.float32)
        w_mean=window.mean(); w_std=window.std()+1e-8
        x = dwt_transform((window-w_mean)/w_std)
        hour = self.df.index[c].hour
        sin_h = np.full(WINDOW_SIZE, np.sin(2*np.pi*hour/24), dtype=np.float32)
        cos_h = np.full(WINDOW_SIZE, np.cos(2*np.pi*hour/24), dtype=np.float32)
        x = np.concatenate([x, sin_h[None], cos_h[None]], axis=0)
        y_power = np.array([np.nan_to_num(
            self.df[a].values[c]/APPLIANCES[a]['max_power'],nan=0.0)
            for a in APPLIANCE_NAMES],dtype=np.float32).clip(0,1)
        y_state = np.array([np.nan_to_num(
            self.df[f'{a}_state'].values[c],nan=0.0)
            for a in APPLIANCE_NAMES],dtype=np.float32)
        return torch.tensor(x), torch.tensor(y_power), torch.tensor(y_state)

print('Loading House 2...')
cached2 = load_clean_df('UK-DALE', 2)
if cached2 is not None: h2_df = cached2
else:
    raw2 = load_ukdale_house(house=2)
    h2_df = preprocess_house(raw2)
    save_clean_df(h2_df, 'UK-DALE', 2)
print(f'House 2: {len(h2_df):,} rows')

h2_loader_1d = DataLoader(CrossHouseDataset1D(h2_df), batch_size=256,
                           shuffle=False, num_workers=2)

for approach, model, loader, name in [
    ('A: 1D DWT', model_a, h2_loader_1d, 'cnn_lstm_1d'),
    ('B: 2D Kurtogram', model_b,
     DataLoader(KurtogramDataset(h2_df, stride=480), batch_size=32,
                shuffle=False, num_workers=0), 'cnn_lstm_2d'),
]:
    model.eval()
    h2_tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
    h2_tracker.reset()
    with torch.no_grad():
        for x, yp, ys in loader:
            pp, ps, pg = model(x.to(DEVICE))
            h2_tracker.update(pp.cpu(), yp, ps.cpu(), ys)
    h2_m = h2_tracker.compute()
    print(f'\n{"="*60}')
    print(f'H2 RESULTS — Approach {approach}')
    print(f'{"="*60}')
    h2_tracker.print_table(h2_m)
    with open(f'{RES_DIR}/{name}_h2_metrics.json','w') as f:
        json.dump(h2_m, f, indent=2)


## 11. Save All

In [ ]:
import zipfile
zip_path = '/kaggle/working/cnn_lstm_complete.zip'
files_to_save = [
    f'{CKPT_DIR}/cnn_lstm_1d_best.pth',
    f'{CKPT_DIR}/cnn_lstm_2d_best.pth',
    f'{RES_DIR}/cnn_lstm_1d_history.json',
    f'{RES_DIR}/cnn_lstm_2d_history.json',
    f'{RES_DIR}/cnn_lstm_1d_h2_metrics.json',
    f'{RES_DIR}/cnn_lstm_2d_h2_metrics.json',
    f'{RES_DIR}/cnn_lstm_comparison.json',
    f'{RES_DIR}/cnn_lstm_comparison.png',
]
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fp in files_to_save:
        if os.path.exists(fp):
            zf.write(fp, os.path.basename(fp))
            print(f'Added: {os.path.basename(fp)}')
size = os.path.getsize(zip_path)/1e6
print(f'\nZIP: cnn_lstm_complete.zip ({size:.1f} MB)')
print('Download from Output tab')
